In [34]:
import numpy as np
import pandas as pd
import os, sys
from pathlib import Path

import statsmodels.formula.api as smf
import statsmodels.api as sm

# BQ 클라이언트
PROJECT_ROOT = str(Path.cwd().parent)
sys.path.insert(0, os.path.join(PROJECT_ROOT, '05_src/02_bigquery'))
from bq_client import query_to_df

print(f"프로젝트 루트: {PROJECT_ROOT}")

프로젝트 루트: /Users/yu_seok/Documents/Document/workspace/03_프로젝트/04_Why-pi


In [35]:
# 제품 기본 정보 + 카테고리 + 브랜드 + 기능성 + SLI
df = query_to_df("""
    SELECT ps.product_code, ps.likes, ps.shares, ps.review_count,
           ps.engagement_score, ps.cp_index, ps.review_density,
           pc.category_1, pc.category_2,
           b.name AS brand, pcore.name AS product_name, pcore.price,
           f.is_whitening, f.is_wrinkle_reduction, f.is_sunscreen, f.is_acne,
           sr.final_soft_landing,
           rc_stats.first_review_date
    FROM daiso.products_stats ps
    JOIN daiso.products_core pcore ON ps.product_code = pcore.product_code
    JOIN daiso.products_category pc ON ps.product_code = pc.product_code
    JOIN daiso.brands b ON pcore.brand_id = b.brand_id
    LEFT JOIN daiso.functional f ON ps.product_code = f.product_code
    LEFT JOIN daiso.sli_results sr ON ps.product_code = sr.product_code
    LEFT JOIN (
        SELECT product_code, MIN(review_date) AS first_review_date
        FROM daiso.reviews_core
        GROUP BY product_code
    ) rc_stats ON ps.product_code = rc_stats.product_code
""")

print(f"제품 수: {len(df)}")
df.head()

/opt/miniconda3/envs/py_study/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


제품 수: 948


,product_code,likes,shares,review_count,engagement_score,cp_index,review_density,category_1,category_2,brand,product_name,price,is_whitening,is_wrinkle_reduction,is_sunscreen,is_acne,final_soft_landing,first_review_date
0,1017957,0,0,33,18.15,6.0500,0.0471,맨케어,남성용면도기,쉬크,쉬크 익스트림 3중날 면도기 4개입,3000.0,<NA>,<NA>,<NA>,<NA>,<NA>,2024-02-17
1,1062561,0,0,17,9.35,1.8700,0.1012,메이크업,치크/하이라이터,파넬,[02 핑크] 파넬 피치마누 글로우밤 하이라이터,5000.0,<NA>,<NA>,<NA>,<NA>,<NA>,2025-07-03
2,1059937,0,0,10,5.50,1.1000,0.0392,맨케어,남성용면도기,KAI,KAI 2중날 면도기 20개입,5000.0,<NA>,<NA>,<NA>,<NA>,<NA>,2025-04-25
3,1059925,0,0,22,12.10,4.0333,0.0812,맨케어,남성용면도기,KAI,KAI 2중날 면도기 10개입,3000.0,<NA>,<NA>,<NA>,<NA>,<NA>,2025-04-29
4,1021252,0,0,6,3.30,1.1000,0.0155,맨케어,남성용면도기,쉬크,쉬크 쿼트로 티타늄 휴대용 면도기 2개입,3000.0,<NA>,<NA>,<NA>,<NA>,<NA>,2024-10-22


In [36]:
# 성분 데이터
ingr = query_to_df("""
    SELECT pi.product_code, pi.ingredient_id,
           d.ingredient_type, d.is_allergic, d.effect
    FROM daiso.products_ingredients pi
    JOIN daiso.ingredients_dic d ON pi.ingredient_id = d.ingredient_id
""")

print(f"성분 매핑 수: {len(ingr)}")
ingr.head()

/opt/miniconda3/envs/py_study/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


성분 매핑 수: 27975


,product_code,ingredient_id,ingredient_type,is_allergic,effect
0,49856,41,Paraben,False,Preservative
1,49856,43,Aldehyde,True,Fragrance
2,49859,54,Paraben,False,Preservative
3,49859,43,Aldehyde,True,Fragrance
4,50371,61,Ketone,False,Fragrance


# 파생변수 계산

In [37]:
# ── age_days ──
today = pd.Timestamp("2026-02-26")
df["first_review_date"] = pd.to_datetime(df["first_review_date"], errors="coerce")
df["age_days"] = (today - df["first_review_date"]).dt.days
df["age_days"] = df["age_days"].fillna(df["age_days"].median())

# ── is_functional ──
func_cols = ["is_whitening", "is_wrinkle_reduction", "is_sunscreen", "is_acne"]
df["is_functional"] = df[func_cols].fillna(False).any(axis=1).astype(int)

# ── log_price ──
df["log_price"] = np.log1p(df["price"])

# ── ingredient_count (제품당 성분 수) ──
ingredient_count = ingr.groupby("product_code").size().rename("ingredient_count")

# ── water_base_ratio (수분 베이스 비율: Water + Polyol) ──
water_base_types = ["Water", "Polyol"]
water_count = ingr[ingr["ingredient_type"].isin(water_base_types)].groupby("product_code").size()
total_count = ingr.groupby("product_code").size()
water_base_ratio = (water_count / total_count).fillna(0).rename("water_base_ratio")

# ── gentle_score (순한 성분 점수 = 1 - 알러지 유발 비율) ──
allergic_count = ingr[ingr["is_allergic"] == True].groupby("product_code").size()
gentle_score = (1 - (allergic_count / total_count).fillna(0)).rename("gentle_score")

# merge
df = df.merge(ingredient_count, on="product_code", how="left")
df = df.merge(water_base_ratio, on="product_code", how="left")
df = df.merge(gentle_score, on="product_code", how="left")
df[["ingredient_count", "water_base_ratio", "gentle_score"]] = (
    df[["ingredient_count", "water_base_ratio", "gentle_score"]].fillna(0)
)

# ── brand_target_enc (Target Encoding: 브랜드별 연착륙 비율) ──
global_mean = df["final_soft_landing"].astype(float).mean()
brand_stats = df.groupby("brand")["final_soft_landing"].agg(["mean", "count"])
smoothing = 10
brand_stats["brand_target_enc"] = (
    (brand_stats["count"] * brand_stats["mean"] + smoothing * global_mean)
    / (brand_stats["count"] + smoothing)
)
df = df.merge(brand_stats[["brand_target_enc"]], left_on="brand", right_index=True, how="left")
df["brand_target_enc"] = df["brand_target_enc"].fillna(global_mean)

# ── category_2_grp (소수 카테고리 그룹화: 완전분리 방지) ──
MIN_CAT_COUNT = 15
cat_counts = df["category_2"].value_counts()
rare_cats = cat_counts[cat_counts < MIN_CAT_COUNT].index.tolist()
df["category_2_grp"] = df["category_2"].where(
    ~df["category_2"].isin(rare_cats), "기타"
)

# 완전분리 검사 (연착륙 라벨 있는 행만)
_labeled = df[df["final_soft_landing"].notna()]
_sep = _labeled.groupby("category_2_grp")["final_soft_landing"].agg(["mean", "count"])
_perfect = _sep[(_sep["mean"] == 0) | (_sep["mean"] == 1)].index.tolist()
if _perfect:
    print(f"완전분리 카테고리 → '기타' 병합: {_perfect}")
    df.loc[df["category_2_grp"].isin(_perfect), "category_2_grp"] = "기타"

print(f"\nis_functional: {df['is_functional'].value_counts().to_dict()}")
print(f"ingredient_count 평균: {df['ingredient_count'].mean():.1f}")
print(f"water_base_ratio 평균: {df['water_base_ratio'].mean():.3f}")
print(f"gentle_score 평균: {df['gentle_score'].mean():.3f}")
print(f"brand_target_enc 범위: [{df['brand_target_enc'].min():.3f}, {df['brand_target_enc'].max():.3f}]")
print(f"\ncategory_2_grp 분포:")
print(_labeled.groupby("category_2_grp")["final_soft_landing"].agg(["mean", "count"]).sort_values("count", ascending=False))

완전분리 카테고리 → '기타' 병합: ['남성용면도기']

is_functional: {0: 709, 1: 239}
ingredient_count 평균: 29.5
water_base_ratio 평균: 0.144
gentle_score 평균: 0.936
brand_target_enc 범위: [0.057, 0.518]

category_2_grp 분포:
                    mean  count
category_2_grp                 
기초스킨케어          0.275132    189
립메이크업           0.017391    115
베이스메이크업         0.235955     89
아이메이크업          0.215909     88
클렌징/필링          0.357143     70
치크/하이라이터        0.065574     61
팩/마스크            0.40625     32
자외선차단제          0.275862     29
립케어             0.555556     18
남성스킨케어               0.2     10
기타                   0.5      4
남성용면도기               0.0      1


# 안정형

In [38]:
df_model = df[df["final_soft_landing"].notna()].copy()
df_model["y"] = df_model["final_soft_landing"].astype(int)

print(f"모델 데이터: {len(df_model)}건, y 분포: {df_model['y'].value_counts().to_dict()}")

formula_stable = """
y ~ C(category_2_grp) + brand_target_enc +
    ingredient_count + water_base_ratio + gentle_score +
    is_functional + log_price
"""

logit_model = smf.logit(formula_stable, data=df_model).fit(method="bfgs", maxiter=100)
print(logit_model.summary())

모델 데이터: 706건, y 분포: {0: 548, 1: 158}
         Current function value: 0.335699
         Iterations: 100
         Function evaluations: 103
         Gradient evaluations: 103
                           Logit Regression Results                           
Dep. Variable:                      y   No. Observations:                  706
Model:                          Logit   Df Residuals:                      689
Method:                           MLE   Df Model:                           16
Date:                Wed, 04 Mar 2026   Pseudo R-squ.:                  0.3686
Time:                        19:16:31   Log-Likelihood:                -237.00
converged:                      False   LL-Null:                       -375.36
Covariance Type:            nonrobust   LLR p-value:                 1.659e-49
                                    coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------

/opt/miniconda3/envs/py_study/lib/python3.12/site-packages/scipy/optimize/_optimize.py:1330: OptimizeWarning: Maximum number of iterations has been exceeded.
  res = _minimize_bfgs(f, x0, args, fprime, callback=callback, **opts)
/opt/miniconda3/envs/py_study/lib/python3.12/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [39]:
# 연착륙 확률 예측
df_model["stability_prob"] = logit_model.predict(df_model)

df_model[["product_name","stability_prob"]].head()

,product_name,stability_prob
12,초초스랩 클린톡스 립 앤 아이 리무버 250 ml,0.384470
18,애교살 라이너 (베이지),0.143351
20,비타민C 클렌징 티슈 50매입,0.946055
23,어퓨 더퓨어 캔디 워터 치크(01 새해복숭아),0.022612
26,바세린 립 테라피 4.8 g 오리지널,0.590549


In [40]:
# 퍼센타일 기반 점수화
df_model["stability_score"] = (
    df_model["stability_prob"].rank(pct=True) * 100
)

In [41]:
df = df.merge(
    df_model[["product_code","stability_prob","stability_score"]],
    on="product_code",
    how="left"
)

In [42]:
top10_stable_products = (
    df.sort_values("stability_score", ascending=False)
      [["product_code","product_name","brand","category_2","stability_score","stability_prob"]]
      .head(10)
)

top10_stable_products

,product_code,product_name,brand,category_2,stability_score,stability_prob
398,67851,녹차 클렌징티슈 30매,다이소,클렌징/필링,100.000000,0.957937
20,58830,비타민C 클렌징 티슈 50매입,다이소,클렌징/필링,99.858357,0.946055
451,1040473,병풀 클렌징티슈 30매,다이소,클렌징/필링,99.716714,0.942294
437,1035081,약산성 ph클렌징폼 150ml,다이소,클렌징/필링,99.575071,0.940696
319,1034516,어성초 캡슐팩,다이소,팩/마스크,99.433428,0.936985
321,1035082,딥 클렌징 폼 150 ml,다이소,클렌징/필링,99.291785,0.929745
302,1034515,화산송이 캡슐팩,다이소,팩/마스크,99.150142,0.920366
389,1018013,스타일리쉬 오토아이브로우펜슬 흑갈색 22호,다이소,아이메이크업,99.008499,0.916070
410,1034517,에그캡슐팩,다이소,팩/마스크,98.866856,0.916065
327,73008,윙크걸 기름종이 파우더,다이소,베이스메이크업,98.725212,0.906328


# 공격형

In [43]:
formula_growth = """
np.log1p(review_density) ~
    C(category_2_grp) + brand_target_enc +
    ingredient_count + water_base_ratio + gentle_score +
    is_functional + log_price
"""

ols_model = smf.ols(formula_growth, data=df).fit(cov_type="HC3")
print(ols_model.summary())

                               OLS Regression Results                               
Dep. Variable:     np.log1p(review_density)   R-squared:                       0.289
Model:                                  OLS   Adj. R-squared:                  0.276
Method:                       Least Squares   F-statistic:                     27.79
Date:                      Wed, 04 Mar 2026   Prob (F-statistic):           4.51e-68
Time:                              19:16:32   Log-Likelihood:                -405.32
No. Observations:                       948   AIC:                             844.6
Df Residuals:                           931   BIC:                             927.2
Df Model:                                16                                         
Covariance Type:                        HC3                                         
                                    coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------

In [44]:
# 공격형 예측값 (log scale, review_density 기준)
df["growth_pred_log"] = ols_model.predict(df)

df[["product_name", "growth_pred_log"]].head()

,product_name,growth_pred_log
0,쉬크 익스트림 3중날 면도기 4개입,0.114103
1,[02 핑크] 파넬 피치마누 글로우밤 하이라이터,0.408777
2,KAI 2중날 면도기 20개입,0.148456
3,KAI 2중날 면도기 10개입,0.114103
4,쉬크 쿼트로 티타늄 휴대용 면도기 2개입,0.114103


In [45]:
df["growth_pred"] = np.expm1(df["growth_pred_log"])

In [46]:
df["growth_score"] = (
    df["growth_pred_log"].rank(pct=True) * 100
)

In [47]:
top10_growth_products = (
    df.sort_values("growth_score", ascending=False)
      [["product_code","product_name","brand","category_2","growth_score","growth_pred_log"]]
      .head(10)
)

top10_growth_products

,product_code,product_name,brand,category_2,growth_score,growth_pred_log
937,1072808,케어존플러스 밀도탄력 랩핑마스크 60 ml,케어존 플러스,팩/마스크,100.000000,1.187828
154,1059839,소미썸 청귤 미백 토닝 패드 50매입,소미썸,팩/마스크,99.894515,1.184349
254,1060487,메디필 락토 모공 리프팅 랩핑 마스크 4 ml 4개입,메디필,팩/마스크,99.789030,1.171735
910,1073256,AHC 에이지리뉴 듀얼 텐션 마스크 타이트닝 1매 28 g,AHC,팩/마스크,99.683544,1.171663
559,1059838,소미썸 가지 필링 토너 패드 50매입,소미썸,팩/마스크,99.578059,1.165791
560,1059837,소미썸 쌀뜬물 클렌징 워터 패드 50매입,소미썸,팩/마스크,99.472574,1.164417
561,1059840,소미썸 당근 카밍 패드 40매입,소미썸,팩/마스크,99.367089,1.152037
562,1059836,소미썸 젤리 미백 아이 패치 30매입,소미썸,팩/마스크,99.261603,1.135643
563,1059835,소미썸 젤리 주름 아이 패치 30매입,소미썸,팩/마스크,99.156118,1.134879
797,1067500,VT PDRN 광채리프팅마스크 1매 27 g+광채에센스 1.5 ml,VT,팩/마스크,99.050633,1.129261


# 내부 브랜드 분석

In [48]:
brand_summary = (
    df.groupby("brand")
      .agg(
          n_products=("product_code","count"),
          mean_stability=("stability_score","mean"),
          mean_growth=("growth_score","mean"),
          median_stability=("stability_score","median"),
          median_growth=("growth_score","median"),
          n_stability=("stability_score","count")
      
      )
      .reset_index()
)


brand_summary["stability_coverage"] = (
    brand_summary["n_stability"] /
    brand_summary["n_products"]
)

brand_summary.head()

,brand,n_products,mean_stability,mean_growth,median_stability,median_growth,n_stability,stability_coverage
0,AHC,7,31.606637,94.665461,29.603399,94.831224,7,1.0
1,KAI,5,NaN,2.647679,NaN,2.478903,0,0.0
2,LG생활건강,1,NaN,6.540084,NaN,6.540084,0,0.0
3,VT,22,76.628895,75.306866,76.770538,78.639241,19,0.863636
4,과일나라,10,83.829084,49.736287,88.314448,44.989451,6,0.6


In [49]:
brand_stable_top10 = (
    brand_summary
        .sort_values("mean_stability", ascending=False)
        [["brand","n_products","mean_stability"]]
        .head(10)
)

brand_stable_top10

,brand,n_products,mean_stability
11,다이소,64,96.545737
57,오릭스,9,95.447187
10,다나한,6,94.702550
45,센카,3,94.003777
63,제이엠솔루션,7,91.444759
42,비알티씨,4,91.265345
35,바세린,16,89.486937
26,마데카21,9,86.229147
13,닥터오라클,10,84.716714
46,셀더마데일리,5,84.532578


In [50]:
brand_growth_top10 = (
    brand_summary
        .sort_values("mean_growth", ascending=False)
        [["brand","n_products","mean_growth"]]
        .head(10)
)

brand_growth_top10

,brand,n_products,mean_growth
47,소미썸,6,99.454993
46,셀더마데일리,5,97.911392
79,팩미인,1,97.784810
0,AHC,7,94.665461
27,마미케어,3,89.873418
62,자민경,3,88.027426
78,파티온,6,86.656118
61,잇츠스킨,5,86.033755
36,바이 리얼베리어,5,85.548523
44,성분에디터,5,85.274262


# 입점 희망 브랜드 예측 모델

In [51]:
def grade(score):
    """백분위 → 등급 변환"""
    if pd.isna(score):
        return "NA"
    if score >= 80:
        return "A (매우 우수)"
    elif score >= 70:
        return "B (우수)"
    elif score >= 60:
        return "C (보통 이상)"
    elif score >= 50:
        return "D (보완 필요)"
    else:
        return "E (재설계 필요)"

In [52]:
# 유효 브랜드 (5개 이상 상품)
brand_summary_valid = (
    df.groupby("brand")
      .agg(
          n_products=("product_code", "nunique"),
          mean_stability=("stability_score", "mean"),
          mean_growth=("growth_score", "mean")
      )
      .reset_index()
      .query("n_products >= 5")
)


def evaluate_new_brand(new_skus_df):
    """신규 브랜드 SKU 평가 — 실제 모델 predict 기반"""
    tmp = new_skus_df.copy()

    # 필수 피처 준비
    tmp["is_functional"] = pd.to_numeric(
        tmp.get("is_functional", 0), errors="coerce"
    ).fillna(0).astype(int)
    tmp["log_price"] = np.log1p(tmp["price"])
    tmp["brand_target_enc"] = global_mean  # 신규 브랜드 → 전체 평균

    # category_2 → category_2_grp 매핑
    valid_grps = df["category_2_grp"].unique()
    tmp["category_2_grp"] = tmp["category_2"].where(
        tmp["category_2"].isin(valid_grps), "기타"
    )

    # 성분 변수 (입력으로 받거나 카테고리 평균으로 대체)
    for col in ["ingredient_count", "water_base_ratio", "gentle_score"]:
        if col not in tmp.columns:
            cat_mean = df.groupby("category_2")[col].mean()
            tmp[col] = tmp["category_2"].map(cat_mean).fillna(df[col].mean())

    # 모델 예측
    tmp["stability_prob"] = logit_model.predict(tmp)
    tmp["growth_pred_log"] = ols_model.predict(tmp)

    # 제품별 점수 (기존 분포 대비 백분위)
    existing_stab = df_model["stability_prob"]
    existing_grow = df["growth_pred_log"]

    tmp["stability_pct"] = tmp["stability_prob"].apply(
        lambda x: (existing_stab < x).mean() * 100
    )
    tmp["growth_pct"] = tmp["growth_pred_log"].apply(
        lambda x: (existing_grow < x).mean() * 100
    )

    # 브랜드 평균
    brand_stability = float(tmp["stability_pct"].mean())
    brand_growth = float(tmp["growth_pct"].mean())

    # 기존 브랜드 분포 대비 백분위
    bs = brand_summary_valid["mean_stability"].dropna()
    bg = brand_summary_valid["mean_growth"].dropna()
    stab_percentile = float((bs < brand_stability).mean() * 100) if len(bs) else np.nan
    grow_percentile = float((bg < brand_growth).mean() * 100) if len(bg) else np.nan

    return {
        "제품별 상세": tmp[["category_2", "price", "stability_prob",
                         "stability_pct", "growth_pred_log", "growth_pct"]].to_dict("records"),
        "브랜드 안정형 점수": round(brand_stability, 2),
        "안정형 백분위": round(stab_percentile, 1),
        "안정형 등급": grade(stab_percentile),
        "브랜드 공격형 점수": round(brand_growth, 2),
        "공격형 백분위": round(grow_percentile, 1),
        "공격형 등급": grade(grow_percentile),
    }

In [53]:
new_brand = pd.DataFrame({
    "category_2": ["기초스킨케어", "자외선차단제", "기초스킨케어"],
    "is_functional": [1, 1, 1],
    "price": [5000, 3000, 5000],
})

result = evaluate_new_brand(new_brand)
for k, v in result.items():
    if k != "제품별 상세":
        print(f"{k}: {v}")
print("\n제품별 상세:")
for item in result["제품별 상세"]:
    print(f"  {item}")

브랜드 안정형 점수: 56.61
안정형 백분위: 56.5
안정형 등급: D (보완 필요)
브랜드 공격형 점수: 73.63
공격형 백분위: 69.7
공격형 등급: C (보통 이상)

제품별 상세:
  {'category_2': '기초스킨케어', 'price': 5000, 'stability_prob': 0.14521901579590002, 'stability_pct': 56.94050991501416, 'growth_pred_log': 0.840240612055502, 'growth_pct': 87.76371308016878}
  {'category_2': '자외선차단제', 'price': 3000, 'stability_prob': 0.1298108100066955, 'stability_pct': 55.949008498583574, 'growth_pred_log': 0.443407614663692, 'growth_pct': 45.358649789029535}
  {'category_2': '기초스킨케어', 'price': 5000, 'stability_prob': 0.14521901579590002, 'stability_pct': 56.94050991501416, 'growth_pred_log': 0.840240612055502, 'growth_pct': 87.76371308016878}


In [54]:
df_score = df[['product_code','stability_score','growth_score']]

In [55]:
df_score.isna().sum()

product_code         0
stability_score    242
growth_score         0
dtype: int64

In [56]:
import os

_tableau_files = {
    "tableau_i": "tableau_ingredients.csv",
    "tableau_r": "tableau_products_reviews.csv",
    "tableau_s": "tableau_search.csv",
}

_loaded = {}
for key, fname in _tableau_files.items():
    if os.path.exists(fname):
        _loaded[key] = pd.read_csv(fname, encoding="utf-8", low_memory=False)
        print(f"{fname}: {len(_loaded[key])}행 로드")
    else:
        print(f"{fname}: 파일 없음 (건너뜀)")

tableau_i = _loaded.get("tableau_i")
tableau_r = _loaded.get("tableau_r")
tableau_s = _loaded.get("tableau_s")

tableau_ingredients.csv: 파일 없음 (건너뜀)
tableau_products_reviews.csv: 파일 없음 (건너뜀)
tableau_search.csv: 파일 없음 (건너뜀)


In [57]:
if tableau_i is not None:
    dfi = pd.merge(tableau_i, df_score, on="product_code", how="left")
if tableau_r is not None:
    dfr = pd.merge(tableau_r, df_score, on="product_code", how="left")
if tableau_s is not None:
    dfs = pd.merge(tableau_s, df_score, on="product_code", how="left")
print("Tableau merge 완료" if any(v is not None for v in [tableau_i, tableau_r, tableau_s]) else "Tableau 파일 없음 — 건너뜀")

Tableau 파일 없음 — 건너뜀


In [58]:
# if tableau_i is not None:
#     dfi.to_csv('tableau_ingredients_score.csv', encoding='utf-8', index=False)
# if tableau_r is not None:
#     dfr.to_csv('tableau_products_reviews_score.csv', encoding='utf-8', index=False)
# if tableau_s is not None:
#     dfs.to_csv('tableau_search_score.csv', encoding='utf-8', index=False)